In [ ]:
# Read in the anndata object
import anndata as ad
from pathlib import Path
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from scipy.stats import spearmanr, pearsonr
import scipy.sparse as sp
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.ticker import ScalarFormatter
import scanpy as sc
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from collections import defaultdict
import warnings
from adjustText import adjust_text

warnings.filterwarnings('ignore')

# Import LeafletFA differential splicing code
# Define module paths
src_path = "/gpfs/commons/home/kisaev/Leaflet-private/src/"

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.append(src_path)

# Import custom modules
import BetaDirichletFactor.differential_splicing as ds

In [ ]:
# Import utility functions - simple direct import
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/')
from utils import *
from figure_plotting import * 

# Import all functions from /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/atse_viz.py
import sys
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/')
from atse_viz import *
import gffutils

mouse_db_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/GENCODE_vM19"
db_mouse = gffutils.FeatureDB(mouse_db_file, keep_order=True)

In [ ]:
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES ONLY
# =============================================================================

# Main parameters to change
MODEL_TRAIN_DATE = "2025-07-04"
MODEL_ANALYSIS_DATE = "2025-07-06"
PARAM_ID = 0

# Base directories
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION"
RESULTS_BASE_DIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/Mouse_Splicing_Foundation/model_train/MOUSE_FOUNDATION/results"

# =============================================================================
# AUTO-GENERATED PATHS - DON'T EDIT BELOW THIS LINE
# =============================================================================

# Core result directories
PARAM_RESULTS_DIR = f"{RESULTS_BASE_DIR}/{MODEL_TRAIN_DATE}/{MODEL_ANALYSIS_DATE}/param_id_{PARAM_ID}"
DATA_DIR = f"{PARAM_RESULTS_DIR}/data"

# Model outputs
MODEL_OUTPUTS_DIR = f"{BASE_DIR}/Leaflet/leafletFAmodel/{MODEL_TRAIN_DATE}"
MODEL_PATH = f"{MODEL_OUTPUTS_DIR}/run_{PARAM_ID}/leafletfa_model.pkl.xz"

# Main data files
SPLICE_ADATA_PATH = f"{DATA_DIR}/splice_adata_PHI_psi_var_obs.h5ad"
PI_VALUES_PATH = f"{DATA_DIR}/PI_values.npy"
FACTOR_LABELS_PATH = f"{DATA_DIR}/factor_label_df.csv"
PERPLEXITY_PATH =  f"{DATA_DIR}/cell_metadata_with_perplexity_param_id_{PARAM_ID}.csv.gz"

DIFF_SPL_PATH = f"{DATA_DIR}/differential_splicing_results.csv"
NMF_SINGLE_PATH = f"{DATA_DIR}/single_rbp_factor_correlations_rho.csv"

# Input data files (usually don't change)
ATSE_ANNDATA_PATH = (
    f"{BASE_DIR}/MODEL_INPUT/062025/"
    "MOUSE_SPLICING_FOUNDATION_Anndata_ATSE_counts_with_waypoints_20250704_232809.h5ad"
)
GE_ANNDATA_scVI_PATH = f"{BASE_DIR}/scVI/ge_adata_with_both_scvi_models_2025-07-05.h5ad"
GE_ANNDATA_NMF_PATH = f"{BASE_DIR}/NMF/ge_adata_with_NMF_model_30_1024_2025-07-04.h5ad"

# Reference files
AGING_GENES_PATH = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/27857814"
RBP_FILE_PATH = "/gpfs/commons/groups/knowles_lab/Karin/VanNostrand_2020_supptable1_41586_2020_2077_MOESM3_ESM.xlsx"

# Output directory for current analysis
OUTPUT_DIR = f"{PARAM_RESULTS_DIR}/analysis_outputs"

# =============================================================================
# LOAD DATA USING CONFIGURED PATHS
# =============================================================================

print(f"Loading data for param_id {PARAM_ID} from {MODEL_TRAIN_DATE}")
print(f"Data directory: {DATA_DIR}")

# Load main datasets
splice_adata = ad.read_h5ad(SPLICE_ADATA_PATH)
pi = np.load(PI_VALUES_PATH)
factor_labels = pd.read_csv(FACTOR_LABELS_PATH)
diff_spl = pd.read_csv(DIFF_SPL_PATH)
nmf_single = pd.read_csv(NMF_SINGLE_PATH)
perplexity_df = pd.read_csv(PERPLEXITY_PATH)

print(f"Loaded splice_adata: {splice_adata.shape}")
print(f"Loaded PI values: {pi.shape}")
print(f"Loaded factor_labels: {factor_labels.shape}")
print(f"Loaded diff_spl: {diff_spl.shape}")
print(f"Loaded nmf_single: {nmf_single.shape}")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

ATSE_FILE_PATH = (
    f"{BASE_DIR}/ATSE_mapper/ATSE_files/MOUSE_FOUNDATION_ATSE_FILE_unanno_also_2025-07-01_00-02-00.txt.gz"
)

# Load splicing data
ge_adata = ad.read_h5ad(GE_ANNDATA_scVI_PATH)
ge_adata_nmf = ad.read_h5ad(GE_ANNDATA_NMF_PATH)
# If ge_adata.obs doesn't have cell_id make it from cell_id_clean
if "cell_id" not in ge_adata.obs.columns:
    ge_adata.obs["cell_id"] = ge_adata.obs["cell_id_clean"]
    ge_adata_nmf.obs["cell_id"] = ge_adata_nmf.obs["cell_id_clean"]

assert np.all(ge_adata.obs["cell_id"].values == splice_adata.obs["cell_id"].values), "Cell IDs in ge_adata and splice_adata do not match or are not in the same order."

# Load aging gene lists
aging_genes_mouse, aging_genes_human = load_aging_genes(AGING_GENES_PATH)
    
# Load RBP genes
rbps = load_rbp_genes(RBP_FILE_PATH)
    
# If ge_adata.var["gene_name"] is not in ge_adata.var_names, then add it
if "gene_name" not in ge_adata.var.columns:
    ge_adata.var["gene_name"] = ge_adata.var_names

rbps["mouse_gene_name"] = rbps["mouse_gene_name"].str.upper()
aging_genes_mouse = [g.upper() for g in aging_genes_mouse]

# if "mouse.id" is in splice_adata.obs rename it to donor_id 
if "mouse.id" in splice_adata.obs.columns:
    print(f"Renaming mouse.id to donor_id in splice_adata.obs")
    splice_adata.obs.rename(columns={"mouse.id": "donor_id"}, inplace=True)
    # Update gene annotations
    ge_adata.var["RBP_gene"] = ge_adata.var["gene_name"].isin(rbps["mouse_gene_name"])
    ge_adata.var["Aging_gene"] = ge_adata.var["gene_name"].isin(aging_genes_mouse)
    rbps = rbps["mouse_gene_name"]
    print(ge_adata.var["RBP_gene"])
    print(ge_adata.var["Aging_gene"])

else:
    splice_adata.var = add_gene_symbols_to_var(splice_adata.var)
    splice_adata.var["RBP_gene"] = splice_adata.var["gene_name"].isin(rbps["gene_name"]) # when running with Human data... 
    splice_adata.var["Aging_gene"] = splice_adata.var["gene_name"].isin(aging_genes_human)
        
    ge_adata.var["RBP_gene"] = ge_adata.var["gene_name"].isin(rbps["gene_name"])
    ge_adata.var["Aging_gene"] = ge_adata.var["gene_name"].isin(aging_genes_human)
    aging_genes = aging_genes_human
    rbps = rbps["gene_name"]

assert np.all(ge_adata.obs_names == ge_adata_nmf.obs_names), "Cell IDs in ge_adata and ge_adata_nmf do not match or are not in the same order."
assert np.all(ge_adata.var_names == ge_adata_nmf.var_names), "Gene names in ge_adata and ge_adata_nmf do not match or are not in the same order."
ge_adata.obsm["X_nmf_standard_mb"] = ge_adata_nmf.obsm["X_nmf_standard_mb"]
ge_adata.varm["nmf_standard_mb_components"] = ge_adata_nmf.varm["nmf_standard_mb_components"]

# Check if predicted_log_norm_tms not in ge_adata.layers then ge_layer_name="log_norm"
ge_layer_name = "log_norm"

# Load ATSE file
atse_df = pd.read_csv(ATSE_FILE_PATH, sep="\t")
atse_df = atse_df[["junction_id", "perfect_match_5_prime", "perfect_match_3_prime", "gene_name"]]
atse_df["junction_annotation"] = "Novel_SS" 
# If perfect_match_5_prime is nonempty then set junction_annotation to "5_prime_annotated"
atse_df.loc[atse_df["perfect_match_5_prime"].notna(), "junction_annotation"] = "5_prime_annotated"
# If perfect_match_3_prime is nonempty then set junction_annotation to "3_prime_annotated"
atse_df.loc[atse_df["perfect_match_3_prime"].notna(), "junction_annotation"] = "3_prime_annotated"
# If perfect_match_5_prime and perfect_match_3_prime are both nonempty then set junction_annotation to "Both_SS_annotated"
atse_df.loc[atse_df["perfect_match_5_prime"].notna() & atse_df["perfect_match_3_prime"].notna(), "junction_annotation"] = "Both_SS_annotated"

# Read in full splice adata 
splice_adata_full=ad.read_h5ad(ATSE_ANNDATA_PATH)

In [ ]:
# Ensure correct matrix
cell_by_junction = splice_adata_full.layers["cell_by_junction_matrix"]
if not sp.issparse(cell_by_junction):
    cell_by_junction = sp.csr_matrix(cell_by_junction)

# Compute per-cell read counts
splice_adata_full.obs["reads_per_cell"] = np.ravel(cell_by_junction.sum(axis=1))

# Compute per-cell non-zero junction count
splice_adata_full.obs["junctions_detected_per_cell"] = np.ravel((cell_by_junction > 0).sum(axis=1))

# Compute per-junction read counts
splice_adata_full.var["reads_per_junction"] = np.ravel(cell_by_junction.sum(axis=0))

# === Plot 1: Reads per Cell by Dataset (with scientific notation) ===
plt.figure(figsize=(5, 5))
sns.histplot(data=splice_adata_full.obs, x="reads_per_cell", hue="dataset", bins=100,
             multiple="layer", element="step")
plt.title("Read Counts per Cell by Dataset", fontsize=14)
plt.xlabel("Reads per Cell", fontsize=12)
plt.ylabel("Cell Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlim(0, np.percentile(splice_adata_full.obs["reads_per_cell"], 99))

# Use scientific notation on x-axis
ax = plt.gca()
ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0))

# Add median lines + labels
ylim = plt.ylim()
line_spacing = (ylim[1] * 0.9) / (len(splice_adata_full.obs["dataset"].unique()) + 1)
for i, ds in enumerate(sorted(splice_adata_full.obs["dataset"].unique())):
    median_val = splice_adata_full.obs.loc[splice_adata_full.obs["dataset"] == ds, "reads_per_cell"].median()
    plt.axvline(median_val, linestyle="--", linewidth=1, alpha=0.7)
    plt.text(plt.xlim()[1] * 0.98, ylim[1] - i * line_spacing,
             f"{ds} median = {int(median_val):.1e}",
             ha="right", va="top", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/reads_per_cell_by_dataset.pdf")

# === Plot 2: Junctions Detected per Cell ===
plt.figure(figsize=(5, 5))
sns.histplot(data=splice_adata_full.obs, x="junctions_detected_per_cell", hue="dataset", bins=100,
             multiple="layer", element="step")
plt.title("Junctions with Reads > 0 per Cell", fontsize=14)
plt.xlabel("Detected Junctions per Cell", fontsize=12)
plt.ylabel("Cell Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlim(0, np.percentile(splice_adata_full.obs["junctions_detected_per_cell"], 99))

ylim = plt.ylim()
line_spacing = (ylim[1] * 0.9) / (len(splice_adata_full.obs["dataset"].unique()) + 1)
for i, ds in enumerate(sorted(splice_adata_full.obs["dataset"].unique())):
    median_val = splice_adata_full.obs.loc[splice_adata_full.obs["dataset"] == ds, "junctions_detected_per_cell"].median()
    plt.axvline(median_val, linestyle="--", linewidth=1, alpha=0.7)
    plt.text(plt.xlim()[1] * 0.98, ylim[1] - i * line_spacing,
             f"{ds} median = {int(median_val)}",
             ha="right", va="top", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/junctions_per_cell_by_dataset.pdf")

In [ ]:
# Step 1: Extract matrix and metadata
cell_by_junction = splice_adata_full.layers["cell_by_junction_matrix"]
if not sp.issparse(cell_by_junction):
    cell_by_junction = sp.csr_matrix(cell_by_junction)

datasets = splice_adata_full.obs["dataset"].unique()
results = []

# Step 2: For each dataset, compute per-junction detection
for ds in sorted(datasets):
    idx = np.where(splice_adata_full.obs["dataset"] == ds)[0]
    submat = cell_by_junction[idx, :]
    counts = np.array((submat > 0).sum(axis=0)).ravel()
    results.append(pd.DataFrame({
        "dataset": ds,
        "cells_per_junction": counts
    }))

# Step 3: Concatenate results
df_cells_per_junction = pd.concat(results, ignore_index=True)

# === Plot 3B: Cells per Junction by Dataset ===
plt.figure(figsize=(5, 5))
sns.histplot(
    data=df_cells_per_junction,
    x="cells_per_junction",
    hue="dataset",
    bins=30,
    multiple="layer",
    element="step",
    log_scale=(True, False)  # log-scale on x-axis only
)

plt.xlabel("Cells with >0 Reads per Junction", fontsize=12)
plt.ylabel("Junction Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlim(0, np.percentile(df_cells_per_junction["cells_per_junction"], 99))

# Add median lines + labels per dataset
ylim = plt.ylim()
line_spacing = (ylim[1] * 0.9) / (len(datasets) + 1)
for i, ds in enumerate(sorted(datasets)):
    median_val = df_cells_per_junction.loc[df_cells_per_junction["dataset"] == ds, "cells_per_junction"].median()
    plt.axvline(median_val, linestyle="--", linewidth=1, alpha=0.7)
    plt.text(plt.xlim()[1] * 0.98, ylim[1] - i * line_spacing,
             f"{ds} median = {int(median_val)}",
             ha="right", va="top", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cells_per_junction_by_dataset.pdf")


In [ ]:
cell_by_junction = splice_adata_full.layers["cell_by_junction_matrix"]
if not sp.issparse(cell_by_junction):
    cell_by_junction = sp.csr_matrix(cell_by_junction)

# Number of cells each junction is detected in (i.e., >0 reads)
splice_adata_full.var["cells_per_junction"] = np.ravel((cell_by_junction > 0).sum(axis=0))
df_junc = splice_adata_full.var[["cells_per_junction", "annotation_status"]].copy()
summary = (
    df_junc.groupby("annotation_status", observed=True)["cells_per_junction"]
    .median()
    .sort_values(ascending=False)
    .reset_index(name="median_cells")
)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5, 5))
sns.histplot(
    data=df_junc,
    x="cells_per_junction",
    hue="annotation_status",
    bins=30,
    multiple="layer",
    element="step",
    log_scale=(True, False)
)

plt.xlabel("Cells with >0 Reads per Junction", fontsize=12)
plt.ylabel("Junction Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlim(0, np.percentile(df_junc["cells_per_junction"], 99))

# Add vertical lines + text for medians
ylim = plt.ylim()
line_spacing = (ylim[1] * 0.9) / (len(summary) + 1)
for i, row in summary.iterrows():
    plt.axvline(row["median_cells"], linestyle="--", linewidth=1, alpha=0.7)
    plt.text(
        plt.xlim()[1] * 0.98,
        ylim[1] - i * line_spacing,
        f"{row['annotation_status']} median = {int(row['median_cells'])}",
        ha="right", va="top", fontsize=9
    )

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cells_per_junction_by_annotation_status.pdf")
plt.show()
